In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings, os
 
from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder, label_binarize
from sklearn.metrics import (
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)
 
warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)
os.makedirs("data", exist_ok=True)

### Load & Prepare

In [2]:
df = pd.read_parquet("D:/project/Global AI Adoption & Workforce Impact/Data/cleaned.parquet")
df.shape

(150000, 52)

In [3]:
TARGET      = "ai_adoption_stage"
STAGE_ORDER = ["none", "pilot", "partial", "full"]

In [4]:
EXCLUDE = [
    "response_id", "company_id", "country",
    "ai_adoption_stage", "adoption_stage_ord",
    "ai_adoption_rate",                          
    "survey_year", "quarter", "time_index",
    "productivity_per_dollar",                   
    "annual_revenue_usd_millions_w",             
    "num_employees_w",
    "ai_investment_per_employee_w",
]

In [5]:
CATEGORICAL = [
    "region", "industry", "company_size",
    "ai_primary_tool", "ai_use_case", "data_privacy_level",
]

In [6]:
FEATURES = [c for c in df.columns if c not in EXCLUDE]

In [7]:
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df[CATEGORICAL] = enc.fit_transform(df[CATEGORICAL])

In [8]:
for col in df[FEATURES].select_dtypes(include=["object", "string"]).columns:
    df[col] = OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=-1
    ).fit_transform(df[[col]])
 
df["y"] = df[TARGET].map({s: i for i, s in enumerate(STAGE_ORDER)})
 
X = df[FEATURES].apply(lambda c: c.cat.codes if hasattr(c, "cat") else c).astype(float)
y = df["y"].copy()
groups = df["company_id"]
 
print(f"  Features: {X.shape[1]}   Samples: {len(X):,}")

  Features: 39   Samples: 150,000


### Train Test Split

In [9]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
 
print(f"  Train: {len(X_train):,}   Test: {len(X_test):,}")
assert len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])) == 0, \
    "Leakage detected — shared company IDs in train/test!"

  Train: 120,043   Test: 29,957


### CLASS WEIGHTS

In [10]:
class_counts  = y_train.value_counts().sort_index()
sample_weights = y_train.map(
    lambda c: len(y_train) / (len(STAGE_ORDER) * class_counts[c])
)

### Train

In [11]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="multi:softprob",
    num_class=4,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
model.fit(X_train, y_train, sample_weight=sample_weights)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=-1, num_class=4, ...)

### Evaluate

In [12]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)
 
macro_f1    = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")
roc_auc     = roc_auc_score(
    label_binarize(y_test, classes=[0, 1, 2, 3]),
    y_prob, multi_class="ovr", average="macro",
)
 
print(f"\n  Macro F1:    {macro_f1:.4f}")
print(f"  Weighted F1: {weighted_f1:.4f}")
print(f"  ROC-AUC:     {roc_auc:.4f}")
print("\n" + classification_report(y_test, y_pred, target_names=STAGE_ORDER))


  Macro F1:    0.8233
  Weighted F1: 0.8871
  ROC-AUC:     0.9787

              precision    recall  f1-score   support

        none       1.00      1.00      1.00      1021
       pilot       0.87      0.90      0.89     13017
     partial       0.91      0.87      0.89     15623
        full       0.40      0.75      0.52       296

    accuracy                           0.89     29957
   macro avg       0.79      0.88      0.82     29957
weighted avg       0.89      0.89      0.89     29957



### Cross-Validation

In [13]:
print("5-fold StratifiedGroupKFold CV...")
cv_model = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    objective="multi:softprob", num_class=4,
    random_state=42, n_jobs=-1, verbosity=0,
)
cv_scores = cross_val_score(
    cv_model, X, y,
    cv=StratifiedGroupKFold(n_splits=5),
    groups=groups,
    scoring="f1_macro",
    n_jobs=-1,
)
print(f"  CV Macro F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

5-fold StratifiedGroupKFold CV...
  CV Macro F1: 0.8212 ± 0.0064


In [14]:
sns.set_theme(style="whitegrid", font_scale=1.0)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
 
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=STAGE_ORDER,
).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion matrix (counts)")
 
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred, normalize="true"),
    display_labels=STAGE_ORDER,
).plot(ax=axes[1], colorbar=False, cmap="Blues", values_format=".2f")
axes[1].set_title("Confusion matrix (normalised)")
 
plt.suptitle(
    f"Module 1 — Adoption Stage Classifier\n"
    f"Macro F1={macro_f1:.3f}  Weighted F1={weighted_f1:.3f}  ROC-AUC={roc_auc:.3f}",
    fontsize=11, y=1.02,
)
plt.tight_layout()
plt.savefig("outputs/m1_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: outputs/m1_confusion_matrix.png")

  Saved: outputs/m1_confusion_matrix.png


### FEATURE IMPORTANCE PLOT

In [15]:
fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
top = fi.head(20)
 
fig, ax = plt.subplots(figsize=(9, 7))
colors = ["#378ADD"] * 5 + ["#7F77DD"] * 5 + ["#888780"] * 10
ax.barh(range(len(top)), top.values[::-1], color=colors[::-1])
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top.index[::-1], fontsize=10)
ax.set_xlabel("Feature importance (gain)")
ax.set_title("Module 1 — Top 20 features by importance")
plt.tight_layout()
plt.savefig("outputs/m1_feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: outputs/m1_feature_importance.png")

  Saved: outputs/m1_feature_importance.png


In [16]:
with open("data/m1_model.pkl", "wb") as f:
    pickle.dump({
        "model":       model,
        "features":    list(FEATURES),
        "stage_order": STAGE_ORDER,
        "encoder":     enc,
    }, f)
print("  Saved: data/m1_model.pkl")

  Saved: data/m1_model.pkl
